# Audio Translate Tool — `add_translation` branch — Colab Test Runner

This notebook clones the **`add_translation`** branch of [`MrNyox/audio_translate_tool`](https://github.com/MrNyox/audio_translate_tool/tree/add_translation), which adds a Stage Two: LLM-based translation (Arabic ⇄ English) on top of Stage One (Qwen3-ASR-1.7B transcription). It downloads both models into `models/`, launches the Flask app, and tunnels it publicly through Cloudflare so you can test it.

**What this notebook does, step by step:**
1. Checks for a GPU (**required** — bitsandbytes 4-bit inference needs CUDA; this branch will not run on CPU).
2. Installs `ffmpeg`.
3. Clones the repo at the `add_translation` branch.
4. Sanity-checks that the frontend isn't missing `pipeline.js` (a historical bug on this repo — harmless if it's already fine).
5. Installs Python dependencies, now including `transformers`, `bitsandbytes`, `accelerate`, and `sentencepiece` for the translation stage.
6. Downloads `Qwen/Qwen3-ASR-1.7B` into `audio_translate_tool/models/Qwen3-ASR-1.7B`.
7. Downloads `unsloth/Qwen2.5-7B-Instruct-bnb-4bit` (pre-quantized, ~5GB) into `audio_translate_tool/models/Qwen2.5-7B-Instruct-bnb-4bit`.
8. Starts the Flask server in the background.
9. Downloads `cloudflared` and opens a public tunnel to the local server.
10. Live-tails logs for errors while you test.
11. Prints the public URL and lets you ping `/api/health`.

> ⚠️ Runtime → Change runtime type → **T4 GPU** (or better). The ASR model (~1.7B, bf16) and the translation LLM (~7B, 4-bit) are loaded **sequentially, not simultaneously** — Stage Two's `unload_previous_model()` frees the ASR model from VRAM before loading the translation model — so a single T4 (16GB) is enough, but expect translation to take a while.


## 1. Check GPU

In [ ]:
!nvidia-smi


Fri Aug 28 00:17:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. System dependencies (ffmpeg)

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!ffmpeg -version | head -n 1


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers


## 3. Clone the repository

In [ ]:
import os

REPO_URL = "https://github.com/MrNyox/audio_translate_tool.git"
REPO_BRANCH = "main"
REPO_DIR = "/content/audio_translate_tool"

if not os.path.isdir(REPO_DIR):
    !git clone -q -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}
else:
    print(f"Repo already cloned, checking out '{REPO_BRANCH}' and pulling latest changes...")
    !cd {REPO_DIR} && git fetch -q origin {REPO_BRANCH} && git checkout -q {REPO_BRANCH} && git pull -q origin {REPO_BRANCH}

%cd {REPO_DIR}
!git branch --show-current
!ls -la


/content/audio_translate_tool
add_translation
total 76
drwxr-xr-x 10 root root 4096 Aug 28 00:17 .
drwxr-xr-x  1 root root 4096 Aug 28 00:17 ..
-rw-r--r--  1 root root 1394 Aug 28 00:17 app.py
-rw-r--r--  1 root root 5837 Aug 28 00:17 config.py
drwxr-xr-x  8 root root 4096 Aug 28 00:17 .git
-rw-r--r--  1 root root    6 Aug 28 00:17 .gitignore
drwxr-xr-x  2 root root 4096 Aug 28 00:17 job_output
drwxr-xr-x  2 root root 4096 Aug 28 00:17 models
drwxr-xr-x  2 root root 4096 Aug 28 00:17 __pycache__
-rw-r--r--  1 root root   26 Aug 28 00:17 README.md
-rw-r--r--  1 root root  184 Aug 28 00:17 requirements.txt
-rw-r--r--  1 root root 9013 Aug 28 00:17 routes.py
drwxr-xr-x  3 root root 4096 Aug 28 00:17 stage_one
drwxr-xr-x  2 root root 4096 Aug 28 00:17 stage_two
drwxr-xr-x  4 root root 4096 Aug 28 00:17 static
drwxr-xr-x  2 root root 4096 Aug 28 00:17 templates


## 4. Sanity-check the frontend bundle

The `add_translation` branch already includes the Stage One fixes (correct `qwen-asr` loading in
`stage_one/asr.py`, matching `config.py`, updated `requirements.txt`) plus the new Stage Two
translation code — so there's nothing to patch here.

This repo *did* previously ship with an empty `static/js/components/pipeline.js`, which silently
breaks the whole frontend (import of `initPipeline`/`handleVisibilityChange` fails, so nothing
binds to the UI). This cell just guards against that specific regression reappearing.

In [ ]:
import os

pipeline_js = "static/js/components/pipeline.js"
size = os.path.getsize(pipeline_js) if os.path.isfile(pipeline_js) else 0

if size == 0:
    raise RuntimeError(
        f"{pipeline_js} is empty (0 bytes) — the frontend will silently fail to load. "
        "This was a known bug on this repo; make sure your branch has the fix committed."
    )

print(f"OK: {pipeline_js} is {size} bytes.")


OK: static/js/components/pipeline.js is 5267 bytes.


## 5. Install Python dependencies

In [ ]:
!pip install -q -r requirements.txt
!pip install -q "huggingface_hub[cli]"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.6/141.6 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 97.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 70.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 416.8/416.8 kB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.3/32.3 MB 72.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.8 MB/s eta 0:00:00
  

## 6. Download Qwen3-ASR-1.7B into `models/`

This downloads the model weights straight from Hugging Face into
`audio_translate_tool/models/Qwen3-ASR-1.7B`, matching the default
`config.MODEL_ID` path so no extra environment variable is strictly needed
(we set it explicitly anyway, for clarity).

In [ ]:
from huggingface_hub import snapshot_download

MODEL_DIR = f"{REPO_DIR}/models/Qwen3-ASR-1.7B"

snapshot_download(
    repo_id="Qwen/Qwen3-ASR-1.7B",
    local_dir=MODEL_DIR,
    # Skip files we don't need for inference (e.g. alternate weight formats).
    ignore_patterns=["*.bin", "*.pt", "*.onnx", "*.md"],
)

print("Model downloaded to:", MODEL_DIR)
!du -sh {MODEL_DIR}


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.22G [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/330 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

Model downloaded to: /content/audio_translate_tool/models/Qwen3-ASR-1.7B
4.4G	/content/audio_translate_tool/models/Qwen3-ASR-1.7B


## 6b. Download the translation model (Qwen2.5-7B-Instruct, 4-bit) into `models/`

Stage Two uses `unsloth/Qwen2.5-7B-Instruct-bnb-4bit` — a pre-quantized (bitsandbytes NF4) 7B
instruct model, ~5GB on disk. Same pattern as the ASR download: pulled straight into `models/`
so `config.TRANSLATION_MODEL_ID` can point at a local folder instead of re-downloading from the
Hub on first use.

In [ ]:
from huggingface_hub import snapshot_download

TRANSLATION_MODEL_DIR = f"{REPO_DIR}/models/Qwen2.5-7B-Instruct-bnb-4bit"

snapshot_download(
    repo_id="unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    local_dir=TRANSLATION_MODEL_DIR,
    ignore_patterns=["*.bin", "*.pt", "*.onnx", "*.gguf", "*.md"],
)

print("Translation model downloaded to:", TRANSLATION_MODEL_DIR)
!du -sh {TRANSLATION_MODEL_DIR}


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

Translation model downloaded to: /content/audio_translate_tool/models/Qwen2.5-7B-Instruct-bnb-4bit
5.2G	/content/audio_translate_tool/models/Qwen2.5-7B-Instruct-bnb-4bit


## 7. Configure environment variables

In [ ]:
import os

os.environ["QWEN3_ASR_MODEL"] = MODEL_DIR
os.environ["QWEN3_ASR_DEVICE"] = "auto"       # will pick cuda:0 automatically if available
os.environ["QWEN3_ASR_DTYPE"] = "auto"        # bfloat16 on GPU, float32 on CPU
os.environ["STAGE_ONE_OUTPUT_ROOT"] = f"{REPO_DIR}/job_output"

# Stage Two — translation model (local folder downloaded in step 6b above).
os.environ["TRANSLATION_MODEL_ID"] = TRANSLATION_MODEL_DIR

# Optional overrides you may want to tweak:
# os.environ["QWEN3_ASR_LANGUAGE"] = "English"     # force a language instead of auto-detect
# os.environ["QWEN3_ASR_MAX_NEW_TOKENS"] = "4096"  # for very long audio
# os.environ["TRANSLATION_MAX_TOKENS"] = "2048"    # max new tokens per translation chunk
# os.environ["TRANSLATION_CHUNK_SIZE"] = "4096"    # input tokens per chunk before splitting

for k in [
    "QWEN3_ASR_MODEL",
    "QWEN3_ASR_DEVICE",
    "QWEN3_ASR_DTYPE",
    "STAGE_ONE_OUTPUT_ROOT",
    "TRANSLATION_MODEL_ID",
]:
    print(f"{k} = {os.environ[k]}")


QWEN3_ASR_MODEL = /content/audio_translate_tool/models/Qwen3-ASR-1.7B
QWEN3_ASR_DEVICE = auto
QWEN3_ASR_DTYPE = auto
STAGE_ONE_OUTPUT_ROOT = /content/audio_translate_tool/job_output
TRANSLATION_MODEL_ID = /content/audio_translate_tool/models/Qwen2.5-7B-Instruct-bnb-4bit


## 8. Launch the Flask app in the background

The app's `if __name__ == "__main__"` block binds to `127.0.0.1:5000` with the Flask dev
server. That's fine for local + tunneled use. We launch it as a background subprocess so the
notebook cell doesn't block, and log its output to `server.log`.

In [ ]:
import subprocess
import time
import requests

LOG_PATH = f"{REPO_DIR}/server.log"

server_process = subprocess.Popen(
    ["python3", "app.py"],
    cwd=REPO_DIR,
    stdout=open(LOG_PATH, "w"),
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
    start_new_session=True,  # survive Ctrl+C / "stop" on other cells
)

print(f"Started Flask server with PID {server_process.pid}")
print("Waiting for it to come up (this can take a while the first time, while the model loads)...")

# Health check - retries for a few minutes since model loading can be slow.
health_url = "http://127.0.0.1:5000/api/health"
deadline = time.time() + 600  # 10 minutes
up = False

while time.time() < deadline:
    if server_process.poll() is not None:
        print("Server process exited early! Check the log below:")
        break
    try:
        resp = requests.get(health_url, timeout=3)
        if resp.ok:
            print("Server is up:", resp.json())
            up = True
            break
    except requests.exceptions.ConnectionError:
        pass
    time.sleep(3)

if not up:
    print("--- server.log tail ---")
    !tail -n 100 {LOG_PATH}


Started Flask server with PID 3002
Waiting for it to come up (this can take a while the first time, while the model loads)...
Server is up: {'data': {'aligner_loaded': False, 'ffmpeg_available': True, 'model_loaded': False, 'status': 'online'}, 'ok': True}


## 9. Tunnel it publicly through Cloudflare

Downloads the `cloudflared` binary and opens a quick, no-account-needed tunnel
(`trycloudflare.com`) pointing at the local Flask server.

In [ ]:
import re
import subprocess
import time

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

CF_LOG_PATH = f"{REPO_DIR}/cloudflared.log"

cf_process = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:5000"],
    cwd=REPO_DIR,
    stdout=open(CF_LOG_PATH, "w"),
    stderr=subprocess.STDOUT,
    start_new_session=True,  # survive Ctrl+C / "stop" on other cells
)

print(f"Started cloudflared with PID {cf_process.pid}")
print("Waiting for the public URL...")

public_url = None
deadline = time.time() + 60

while time.time() < deadline:
    try:
        with open(CF_LOG_PATH) as f:
            log_contents = f.read()
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_contents)
        if match:
            public_url = match.group(0)
            break
    except FileNotFoundError:
        pass
    time.sleep(2)

if public_url:
    print(f"\nYour app is live at: {public_url}\n")
else:
    print("Could not find the tunnel URL yet, check the log below:")
    !cat {CF_LOG_PATH}


Started cloudflared with PID 3078
Waiting for the public URL...

Your app is live at: https://agricultural-dee-translate-additionally.trycloudflare.com



## 10. Live log & error monitor

This cell scans `server.log` (Flask/model/ASR output) and `cloudflared.log` (tunnel output) for
anything that looks like a real problem — tracebacks, `ModuleNotFoundError`, CUDA OOM, connection
refused, 5xx responses, etc. — and prints it out immediately as it happens.

It first sweeps whatever is already in the logs, then tails both files live, checking for new
lines every couple of seconds. It prints a 🟢 heartbeat every 30s so you know it's still watching,
and 🔴 flags anything error-like the moment it shows up.

**This cell runs forever (by design) so you can leave it running while you test uploads in the
browser.** Stop it whenever you're done — click the cell's stop/square button in Colab, which
raises a `KeyboardInterrupt` that this cell catches cleanly.


In [ ]:
import re
import time
from datetime import datetime

# Patterns that typically indicate a real problem (case-insensitive).
ERROR_PATTERNS = [
    r"traceback \(most recent call last\)",
    r"\berror\b",
    r"\bexception\b",
    r"\bcritical\b",
    r"\bfailed\b",
    r"\bfailure\b",
    r"cuda out of memory",
    r"out of memory",
    r"modulenotfounderror",
    r"importerror",
    r"attributeerror",
    r"connection refused",
    r"address already in use",
    r"502 bad gateway",
    r"503 service unavailable",
    r"permission denied",
    r"no such file or directory",
]
ERROR_RE = re.compile("|".join(ERROR_PATTERNS), re.IGNORECASE)

# Lines that look scary but are actually benign noise from these specific tools.
IGNORE_PATTERNS = [
    r"^\s*\*\s*(Serving Flask app|Debug mode|Running on|Restarting with|Debugger)",
    r"werkzeug.*\b(GET|POST)\b",
]
IGNORE_RE = re.compile("|".join(IGNORE_PATTERNS), re.IGNORECASE)

LOG_FILES = {
    "server.log": LOG_PATH,
    "cloudflared.log": CF_LOG_PATH,
}


def _scan_new_lines(path, offset):
    """Read any lines appended to `path` since byte `offset`. Returns (new_offset, lines)."""
    try:
        with open(path, "r", errors="replace") as f:
            f.seek(offset)
            new_data = f.read()
            new_offset = f.tell()
        return new_offset, new_data.splitlines()
    except FileNotFoundError:
        return offset, []


# First, sweep whatever is already in the logs so nothing prior gets missed.
offsets = {}
print("Scanning existing logs for prior errors...\n")
found_any = False
for name, path in LOG_FILES.items():
    new_offset, lines = _scan_new_lines(path, 0)
    offsets[name] = new_offset
    for line in lines:
        if not line.strip() or IGNORE_RE.search(line):
            continue
        if ERROR_RE.search(line):
            found_any = True
            print(f"[{name}] {line}")
if not found_any:
    print("(no errors found in existing logs)")

print("\nLive-tailing logs for new errors. Stop this cell (Colab: click the stop/square button) when you are done testing.\n")

HEARTBEAT_EVERY = 30  # seconds
last_heartbeat = time.time()

try:
    while True:
        any_new_error = False

        for name, path in LOG_FILES.items():
            new_offset, lines = _scan_new_lines(path, offsets[name])
            offsets[name] = new_offset

            for line in lines:
                if not line.strip() or IGNORE_RE.search(line):
                    continue
                if ERROR_RE.search(line):
                    any_new_error = True
                    ts = datetime.now().strftime("%H:%M:%S")
                    print(f"\U0001F534 [{ts}] [{name}] {line}")

        if not any_new_error and time.time() - last_heartbeat > HEARTBEAT_EVERY:
            ts = datetime.now().strftime("%H:%M:%S")
            print(f"\U0001F7E2 [{ts}] still watching, no new errors...")
            last_heartbeat = time.time()

        # Also bail out (with a clear message) if either background process died.
        if server_process.poll() is not None:
            print("\n\U0001F534 Flask server process has exited! Check server.log above for the cause.")
            break
        if cf_process.poll() is not None:
            print("\n\U0001F534 cloudflared process has exited! Check cloudflared.log above for the cause.")
            break

        time.sleep(2)

except KeyboardInterrupt:
    print("\nStopped log monitor.")


Scanning existing logs for prior errors...

[cloudflared.log] 2026/08/27 22:02:39 failed to sufficiently increase receive buffer size (was: 208 kiB, wanted: 7168 kiB, got: 416 kiB). See https://github.com/quic-go/quic-go/wiki/UDP-Buffer-Sizes for details.

Live-tailing logs for new errors. Stop this cell (Colab: click the stop/square button) when you are done testing.


Stopped log monitor.


## 11. Try it out

Open the printed `trycloudflare.com` URL in your browser. Upload an audio or video file and hit
**Process Media** — it will extract the audio, run it through Qwen3-ASR-1.7B, and give you back
the transcript (plus a muted video download if you uploaded a video).

You can also poke the API directly from the notebook:

In [ ]:
import requests

# Simple health check through the tunnel.
if public_url:
    r = requests.get(f"{public_url}/api/health")
    print(r.status_code, r.json())


200 {'data': {'aligner_loaded': False, 'ffmpeg_available': True, 'model_loaded': False, 'status': 'online'}, 'ok': True}


## 12. Cleanup (optional)

Run this cell when you're done to stop the Flask server and the tunnel.

In [ ]:
%cd models
%ls
!curl -L -o IBMPlexSansArabic-Regular.ttf \
  https://github.com/google/fonts/raw/main/ofl/ibmplexsansarabic/IBMPlexSansArabic%5Bwght%5D.ttf

/content/audio_translate_tool/models
Qwen2.5-7B-Instruct-bnb-4bit/  Qwen3-ASR-1.7B/
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  298k    0  298k    0     0   833k      0 --:--:-- --:--:-- --:--:--  831k


In [ ]:
import os
import signal

for proc, name in [(server_process, "Flask server"), (cf_process, "cloudflared tunnel")]:
    try:
        # These were started with start_new_session=True (their own process
        # group), so send the signal to the whole group, not just the PID.
        os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
        proc.wait(timeout=10)
        print(f"Stopped {name}.")
    except ProcessLookupError:
        print(f"{name} was already stopped.")
    except Exception as e:
        print(f"Could not stop {name} cleanly: {e}")
